> ⏱ ~5 min

# Scenario

**Section 1.** Before you build a multi-agent concierge, you confirm that the Microsoft Agent Framework (MAF) can reach the Cohere `command-a` deployment in your Microsoft Foundry project. This is the same model a Foundry-hosted agent used; we are just calling it from a different runtime.

A **chat client** is the small piece of code that turns a Python message into an HTTP request the model understands. MAF ships an `OpenAIChatClient` that handles all of that, so you only need to give it the account endpoint (`AZURE_AI_ENDPOINT`/openai/v1) and the deployment name.


## 2. What you will do

1. Load Lab 0 environment settings with `python-dotenv`.
2. Create an `OpenAIChatClient` pointed at `command-a`.
3. Wrap it in a one-shot `Agent` and send a tiny prompt.
4. Confirm the response text comes back from the Cohere deployment.

If this notebook fails, the multi-agent notebooks would fail for the same reason but with more moving parts. Fixing the connection here keeps later debugging short.

---


In [ ]:
# Suppress experimental/deprecation warnings to keep the learning output clean.
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
from dotenv import load_dotenv

# Locate this lab folder portably (do not hardcode the repo name; learners may fork+rename).
LAB_DIR = Path.cwd().resolve()
if LAB_DIR.name != 'lab-1-foundry-maf':
    for _p in (LAB_DIR, *LAB_DIR.parents):
        _candidate = _p / 'cohere' / 'lab-1-foundry-maf'
        if _candidate.exists():
            LAB_DIR = _candidate
            break
COHERE_DIR = LAB_DIR.parent
load_dotenv(COHERE_DIR / '.env')

# Lab 0's setup-env.sh writes these values; sample.env documents each key.
ACCOUNT_ENDPOINT = os.environ['AZURE_AI_ENDPOINT']
DEPLOYMENT = os.getenv('COMMAND_A_DEPLOYMENT', 'command-a')
print(f'Account endpoint: {ACCOUNT_ENDPOINT}')
print(f'Chat deployment : {DEPLOYMENT}')


## 3. Open a MAF chat client

`OpenAIChatClient` is MAF's general-purpose OpenAI Responses client. Pointed at your Foundry account's `/openai/v1` endpoint with a Microsoft Entra credential, it can reach any model the account exposes — including Cohere — using the same auth a Foundry-hosted agent would use.

The next code cell creates the client. No request is sent yet; this is the equivalent of picking up the phone before dialing.


In [ ]:
import sys
sys.path.insert(0, str(LAB_DIR))

from agents import make_chat_client

client = make_chat_client(endpoint=ACCOUNT_ENDPOINT, model=DEPLOYMENT)
print('Client type :', type(client).__name__)


## 4. Run a one-shot agent

An **Agent** in MAF is a model plus instructions plus optional tools. The smallest possible test is an agent with one short instruction and no tools. If it answers, the chat client is wired correctly.

The next cell creates that tiny agent and sends a single question.


In [ ]:
from agent_framework import Agent

smoke_agent = Agent(
    client,
    instructions='You are a concise enterprise travel concierge.',
    name='smoke_test_concierge',
)

# Notebooks run an event loop already, so `await` works directly at top level.
response = await smoke_agent.run('In one sentence, say what you help employees book.')
print(response.text)


## 5. What you confirmed

1. `OpenAIChatClient` reaches the Foundry account with the same Microsoft Entra credential a Foundry-hosted agent would use.
2. The Cohere `command-a` deployment answers chat completions through that client.
3. MAF's `Agent.run` returns an `AgentResponse` whose `.text` field is the plain answer.

Next, `02-build-multi-agent.ipynb` wires four agents together: a concierge that delegates to flight, hotel, and car specialists.
